In [ ]:
!pip install seaborn
!pip install scikit-learn

<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; font-weight: bold; margin-top: 25px;">

# Introduction to Unsupervised Learning: customer segmentation
    
</div>

> Example to explain the concept of customer segmentation
>
> **Objective:** You are the owner of a shopping center and want to understand the behavior of your customers in order to give this information to the marketing team and plan the strategy accordingly.

> * For this example, Hierarchical and K-means clustering methods are used. 
> * Also, dimensionality reduction is used to plot the data in 2D dimension plot. 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Import data
df=pd.read_csv("Data/Mall_Customers.csv")
df.head()

In [ ]:
# Let's check for missing values
df.isnull().sum()

In [ ]:
df.dtypes

<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; margin-top: 25px;">

Gender is an Object type. We will need to transform the string feature into discrete numerical values. 
</div>

<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; font-weight: bold; margin-top: 25px;">

## Exploratory Data Analysis (EDA)
    
</div>

> Gender distribution 
> 
> Gender distribution by age

In [ ]:
# Gender distribution

import seaborn as sns
sns.countplot(x='Gender', data=df)
plt.title('Distribution of Gender')

In [ ]:
df.describe()

<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; margin-top: 25px;">

 Gender distribution by age
    
</div>


In [ ]:
# Histogram of plots to get the data distribution
plt.hist('Age', data=df[df['Gender'] == 'Male'], alpha=0.2, color='red', label='Male')
plt.hist('Age', data=df[df['Gender'] == 'Female'], alpha=0.2, color='blue',label='Female')
plt.title('Age distribution by gender')
plt.xlabel('Age')
plt.legend()

<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; margin-top: 25px;">

 Age distribution by gender and income distribution by gender    
</div>


In [ ]:
# We can also get the distribution using the kernel density estimation (KDE)
sns.kdeplot(data=df[df['Gender'] == 'Male']['Age'], 
            fill=True, color='green', label='Male', alpha=0.3)
sns.kdeplot(data=df[df['Gender'] == 'Female']['Age'], 
            fill=True, color='blue', label='Female', alpha=0.3)

# Add a descriptive title and axis labels
plt.title('Age Distribution by Gender (Density Plot)', fontsize=14, fontweight='bold')
plt.xlabel('Age', fontsize=12)
plt.ylabel('Density', fontsize=12)

# Add a legend
plt.legend(title="Gender", fontsize=10)


In [ ]:
sns.kdeplot(data=df[df['Gender'] == 'Male']['Annual Income (k$)'], 
            fill=True, color='green', label='Male', alpha=0.3)
sns.kdeplot(data=df[df['Gender'] == 'Female']['Annual Income (k$)'], 
            fill=True, color='blue', label='Female', alpha=0.3)

# Add a descriptive title and axis labels
plt.title('Annual Income (k$) Distribution by Gender (Density Plot)', fontsize=14, fontweight='bold')
plt.xlabel('Annual Income (k$)', fontsize=12)
plt.ylabel('Density', fontsize=12)

# Add a legend
plt.legend(title="Gender", fontsize=10)


How does KDE work?

<img src="Figures/kde.png" width="600">

<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; margin-top: 25px;">

 Boxplot Age - Gender
    
</div>

In [ ]:
sns.set(style="whitegrid")
sns.boxplot(x="Gender", y="Age", data=df)

<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; margin-top: 25px;">

Boxplot with 'Annual Income(k$)'
    
</div>

In [ ]:
sns.set(style="whitegrid")
sns.boxplot(x="Gender", y="Annual Income (k$)", data=df)


<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; margin-top: 25px;">

Boxplot with 'Spending Score (1-100)'
    
</div>

In [ ]:
sns.set(style="whitegrid")
sns.boxplot(x="Gender", y="Spending Score (1-100)", data=df)

> ✏️ **Question 1:** Looking at the boxplots, are there clear differences between men and women in age, income or spending?

*Your answer:*



<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; margin-top: 25px;">

Variables correlation
    
</div>

In [ ]:
# Let's check variables correlation
sns.heatmap(df.corr(), annot=True)

<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; margin-top: 25px;">

Error! We have string data in our Dataframe. 
    
</div>

> * If the model you're using doesn't assume an ordinal relationship (like most modern ML models), use **One-Hot Encoding**.
>
> * If dimensionality is a concern or you're using algorithms like decision trees that are robust to ordinal values, **Label Encoding** is fine.

In [ ]:
# TODO: which column must be encoded?
df = pd.get_dummies(df, columns=[___], drop_first=True, dtype=int)

In [ ]:
df

In [ ]:
# Let's check variables correlation
sns.heatmap(df.corr(), annot=True)

> **Note:** `CustomerID` has a 0.98 correlation with income only because the file is sorted by income. An identifier is not a feature, so we drop it before modelling.

> ✏️ **Question 2:** Is the correlation between `CustomerID` and income a real relationship? Should we use `CustomerID` as a feature?

*Your answer:*




<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; margin-top: 25px;">

## Example 1. Hierarchical Clustering: groups of clients which are similar regarding Annual Income vs. Age
    
</div>

> * **Agglomerative Clustering** is a type of hierarchical clustering algorithm used to group similar data points together. It is a bottom-up approach, where each data point starts as its own individual cluster, and pairs of clusters are merged as we move up the hierarchy.

> Bottom-up Approach:
> * Initially, each data point is treated as its own cluster.
> * The algorithm iteratively merges the two closest clusters at each step.
> * This process continues until all data points are merged into a single cluster or until a predefined number of clusters is reached.

> ``AgglomerativeClustering(n_clusters=5)``: Hierarchical clustering is performed, and the n_clusters argument specifies the number of clusters you want to identify. You can modify this number based on your analysis or the dendrogram.

> ``fit_predict()``: This method fits the model to the scaled data and assigns each data point to a cluster.

In [ ]:
# create a subset/specific dataframe for this exercise
df_age_income = df[['Age', 'Annual Income (k$)']]

In [ ]:
df_age_income

<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; margin-top: 25px;">

Scale the data and Apply Agglomerative clustering
    
</div>



In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler


# Step 1: Scale the data
scaler = StandardScaler()
df_age_income_scaled = ___  # TODO: fit the scaler and transform df_age_income

# Step 2: Apply Agglomerative Clustering (Hierarchical Clustering)
agg_clustering = AgglomerativeClustering(n_clusters=5)  # You can adjust the number of clusters as needed -- (default) n_clusters=2
y_hierarchical = agg_clustering.fit_predict(df_age_income_scaled)


In [ ]:
y_hierarchical

In [ ]:
df_age_income_scaled

In [ ]:
# Step 3: Plot the results
plt.figure(figsize=(8, 6))
plt.scatter(df_age_income_scaled[:, 0], df_age_income_scaled[:, 1], c=y_hierarchical, cmap='viridis', s=50)
plt.title('Hierarchical Clustering')
plt.xlabel('Age (scaled)')
plt.ylabel('Annual Income (scaled)')
plt.show()


<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; margin-top: 25px;">

Plot the results with non-scale data to understand better the groups
    
</div>


In [ ]:

# Step 3: Plot the results
plt.figure(figsize=(8, 6))
plt.scatter(df_age_income.loc[:, "Age"], df_age_income.loc[:, "Annual Income (k$)"], c=y_hierarchical, cmap='viridis', s=50)
plt.title('Hierarchical Clustering')
plt.xlabel('Age')
plt.ylabel('Annual Income (k$)')
plt.show()

> ✏️ **Question 3:** Why do we scale the data before clustering? Describe the groups you get in terms of age and income.

*Your answer:*



<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; margin-top: 25px;">

## Visualizing the Dendrogram
    
</div>

> The dendrogram helps to visualize how clusters are merged at each level, and the vertical distance between branches indicates how similar the clusters are.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

# Step 4: Create the linkage matrix for the dendrogram
linked = linkage(df_age_income_scaled, 'ward')  # same scaled data as the clustering above

# Step 5: Plot the dendrogram
plt.figure(figsize=(20, 7))
dendrogram(linked)
plt.title('Dendrogram for Hierarchical Clustering')
plt.xlabel('Sample Index')
plt.ylabel('Ward distance')
plt.show()

> ✏️ **Question 4:** Where would you cut the dendrogram, and how many clusters would that give?

*Your answer:*





<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; margin-top: 25px;">

## Example 2. K-means: Find K groups of clients which are similar regarding Spending Score vs. Age
    
</div>

> * To set the number of clusters we want to have as a result, we will use the **Elbow method**


In [ ]:
sns.scatterplot(x='Age', y='Spending Score (1-100)', data=df)
plt.title('Age to Spending Score')

<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; font-weight: bold; margin-top: 25px;">

Elbow method
    
</div>

> * The Elbow Method is a commonly used technique in clustering analysis to determine the optimal number of clusters (k) in a dataset.
> * On the graph, the **x-axis** represents the **number of clusters (k)**, and the **y-axis** represents the within-cluster sum of squared errors **(WCSS)**.
> * The Elbow Method helps balance between:
    * Underfitting: Too few clusters (large WSS).
    * Overfitting: Too many clusters (small WSS but with less meaningful groupings).
> * By choosing the "elbow point," you aim for a model that captures the structure of the data efficiently without unnecessary complexity.

<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; margin-top: 25px;">

Scale the data
    
</div>

In [ ]:
from sklearn.preprocessing import StandardScaler

# create a subset/specific dataframe for this exercise
df_age_spending = df[['Age', 'Spending Score (1-100)']]

# Scale the data
scaler = StandardScaler()
df_age_spending_scaled = scaler.fit_transform(df_age_spending)

In [ ]:
df_age_spending_scaled

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans


wcss = []
K = range(1,15)

for k in K:
    km = KMeans(n_clusters=k, random_state=0)
    km = km.fit(df_age_spending_scaled)
    wcss.append(___)  # TODO: store the WCSS of this model (hint: km.inertia_)
plt.plot(K, wcss, 'bx-')
plt.xlabel('Number of centroids')
plt.ylabel('WCSS')
plt.title('Elbow Method For Optimal k')
plt.show()

> ✏️ **Question 5:** Where is the elbow? Which k would you choose, and why is it not always obvious?

*Your answer:*



> * ``fit()``: Fits the k-means model to the data. This means that the k-means algorithm calculates the positions of the cluster centroids and assigns them to the data instances based on these positions.
> * ``predict()``: Assigns each data point to one of the clusters already calculated by the model. In other words, after the centroids of the clusters are calculated, the algorithm assigns each point to its closest centroid.

In [ ]:
# TODO: choose k from the elbow plot
kmeans = KMeans(n_clusters=___, random_state=0).fit(df_age_spending_scaled)
y_kmeans = kmeans.predict(df_age_spending_scaled)

In [ ]:
y_kmeans

In [ ]:
plt.scatter(df['Age'], df['Spending Score (1-100)'], c=y_kmeans, s=50, alpha=0.5,cmap='viridis')
centers = scaler.inverse_transform(kmeans.cluster_centers_)  # back to original units
plt.scatter(centers[:, 0], centers[:, 1], c='red', marker='X', s=200, label='Centroids')
plt.legend()
plt.xlabel('Age')
plt.ylabel('Spending Score (1-100)')

#### Conclusions

> ✏️ **Question 6:** Describe each group so the marketing team can use it (age, spending).

*Your answer:*





<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; margin-top: 25px;">

## Example 3. K-means: Find K groups of clients which are similar regarding Spending Score vs Annual Income
    
</div>

> * To set the number of clusters we want to have as a result, we will use the **Elbow method**


**Which groups do you detect?**

In [ ]:
sns.scatterplot(x='Annual Income (k$)', y='Spending Score (1-100)', data=df)
plt.title('Annual Income(k$) to Spending Score')

In [ ]:
# create a subset/specific dataframe for this exercise
df_income_spending = df[['Annual Income (k$)', 'Spending Score (1-100)']]


# Scale the data
scaler = StandardScaler()
df_income_spending_scaled = scaler.fit_transform(df_income_spending)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import time

wcss = []
K = range(1,15)

for k in K:
    km = KMeans(n_clusters=k, random_state=0)
    km = km.fit(df_income_spending_scaled)
    wcss.append(km.inertia_)
    


# Plot the Elbow
plt.plot(K, wcss, 'bx-')
plt.xlabel('Number of centroids')
plt.ylabel('WCSS')
plt.title('Elbow Method For Optimal k')
plt.show()


In [ ]:
# Choose Optimal K and run
# TODO: choose k from the elbow plot
kmeans = KMeans(n_clusters=___, random_state=0).fit(df_income_spending_scaled)
y_kmeans = kmeans.predict(df_income_spending_scaled)

In [ ]:
y_kmeans

In [ ]:
plt.scatter(df['Annual Income (k$)'], df['Spending Score (1-100)'], c=y_kmeans, s=50, alpha=0.5,cmap='viridis')
centers = scaler.inverse_transform(kmeans.cluster_centers_)  # back to original units
plt.scatter(centers[:, 0], centers[:, 1], c='red', marker='X', s=200, label='Centroids')
plt.legend()
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')

### Conclusions

> ✏️ **Question 7:** Describe the groups in terms of income and spending. Which group would you target first, and why?

*Your answer:*




<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; margin-top: 25px;">

#  Dimensionality reduction with linear unsupervised method, Principal Component Analysis (PCA) 
    
</div>


> * Principal Component Analysis (PCA) is a statistical technique used for dimensionality reduction. It transforms a high-dimensional dataset into a lower-dimensional one by identifying the most important directions (called principal components) that capture the maximum variance in the data. This allows for a more efficient representation of the data while preserving its essential structure, making it easier to visualize and analyze.
> * In simpler terms, PCA reduces the number of variables in the data while retaining as much information as possible.
> * We use the overall dataset to cluster clients, considering all the information given

In [ ]:
df.head()

In [ ]:
df = df.drop(columns=["CustomerID"])
df

## Dimension reduction: PCA

> * Reduce the dataset to 2 principal components that capture the maximum information of the original attributes.
> * By applying PCA, we transform the data into a 2D space, allowing us to visually observe the structure and grouping of the data points.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = StandardScaler()
scaled_data = scaler.fit_transform(df)  # Scaling the data

pca = PCA(n_components=2)
results_pca = pca.fit_transform(___)  # TODO: which data should PCA use?

print("Explained variance ratio:", pca.explained_variance_ratio_.round(2))
pd.DataFrame(pca.components_, columns=df.columns, index=['PC1', 'PC2']).round(2)  # loadings


> ✏️ **Question 8:** How much information do the two components keep? Looking at the loadings, what do PC1 and PC2 represent? What would happen if we skipped the scaling?

*Your answer:*



In [ ]:
plt.scatter(results_pca[:, 0], results_pca[:, 1],alpha=0.4)
plt.show()

Calculate for this 2 features, the optimal number of clusters

In [ ]:
wcss = []
K = range(1,15)

for k in K:
    km = KMeans(n_clusters=k, random_state=0)
    km = km.fit(results_pca)
    wcss.append(km.inertia_)
    
plt.plot(K, wcss, 'bx-')
plt.xlabel('Number of centroids')
plt.ylabel('WCSS')
plt.title('Elbow Method For Optimal k')
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score

# Silhouette score: a second opinion on k (higher is better)
for k in range(2, 9):
    labels = KMeans(n_clusters=k, random_state=0).fit_predict(results_pca)
    print(k, round(silhouette_score(results_pca, labels), 3))


> ✏️ **Question 9:** Do the elbow method and the silhouette score agree? Which k do you choose?

*Your answer:*



In [ ]:
kmeans = KMeans(n_clusters=___, random_state=0).fit(results_pca)  # TODO: k from elbow + silhouette
y_kmeans = kmeans.predict(results_pca)

plt.scatter(results_pca[:, 0], results_pca[:, 1], c=y_kmeans, s=50, alpha=0.5,cmap='viridis')
centers = kmeans.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], c='red', marker='X', s=200)
plt.xlabel('PC 1')
plt.ylabel('PC 2')

## Adding the resulting Clusters to the Original Data

> * **Clustering results in labels**: Through the clustering process, we have generated labels for each data point.
> * **Supervised learning with cluster labels**: These labels can now be used to train a supervised classification model. Clustering is a common way to create labels when none exist, but those labels are only as good as the clusters.
>  * **Predicting customer groups**: When a new customer arrives, we can predict which group they belong to based on their input data, without needing any prior study or manual grouping.


In [ ]:
y_kmeans

In [ ]:
df_labels = df.copy()
df_labels['cluster'] = y_kmeans

In [ ]:
df_labels.head()



<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; margin-top: 25px;">

 ## Plot Age vs Annual Income, coloring the data points for each Cluster
    
</div>

In [ ]:
plt.figure(figsize=(8, 6))

# Scatter plot with color depending on the 'cluster' column
sns.scatterplot(data=df_labels, x='Age', y='Annual Income (k$)', hue='cluster', palette='viridis', s=100)

# Customize the plot
plt.title('Age vs Annual Income by Cluster')
plt.xlabel('Age')
plt.ylabel('Annual Income (k$)')
plt.legend(title='Cluster', loc='upper left')

# Show plot
plt.show()



<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; margin-top: 25px;">

 ## Plot Annual Income (k$) vs Spending Score, coloring the data points for each Cluster
    
</div>

In [ ]:
plt.figure(figsize=(8, 6))

# Scatter plot with color depending on the 'cluster' column
sns.scatterplot(data=df_labels, y='Spending Score (1-100)', x='Annual Income (k$)', hue='cluster', palette='viridis', s=100)

# Customize the plot
plt.title('Annual Income (k$) vs Spending Score by Cluster')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.legend(title='Cluster', loc='upper left')

# Show plot
plt.show()



<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; margin-top: 25px;">

 ## Plot Age vs Spending Score, coloring the data points for each Cluster
    
</div>

In [ ]:
plt.figure(figsize=(8, 6))

# Scatter plot with color depending on the 'cluster' column
sns.scatterplot(data=df_labels, y='Spending Score (1-100)', x='Age', hue='cluster', palette='viridis', s=100)

# Customize the plot
plt.title('Age vs Spending Score by Cluster')
plt.xlabel('Age')
plt.ylabel('Spending Score (1-100)')
plt.legend(title='Cluster', loc='upper left')

# Show plot
plt.show()

<div style="background-color: #f0f0f0; padding: 25px; border-radius: 5px; margin-top: 25px;">

 # Additional: t-SNE (t-distributed Stochastic Neighbor Embedding)
> * **t-SNE** is a dimensionality reduction algorithm used to visualize high-dimensional data in a low-dimensional space, typically two or three dimensions.
> * t-SNE **converts pairwise distances into probability distributions** in high- and low-dimensional spaces, then optimizes the embedding by minimizing their divergence—focusing strongly on preserving local neighborhoods rather than global geometry.
> * Unlike PCA’s linear projection, t-SNE **uses a nonlinear optimization (KL divergence)** that can uncover complex, curved manifolds and cluster structure that PCA cannot represent.
> * t-SNE is **designed for visualization** (typically 2–3D), often exaggerating cluster separation; PCA is a linear dimensionality reduction technique that preserves global variance and is suitable both for visualization and as a preprocessing step.
    
</div>

In [ ]:
df

In [ ]:
df_scaled = StandardScaler().fit_transform(df.values)

kmeans = KMeans(n_clusters=5, random_state=0)
y_kmeans = kmeans.fit_predict(df_scaled)

In [ ]:
from sklearn.manifold import TSNE

# t-SNE (components mean dimensions)
tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto', init='pca', random_state=42)
X_embedded = tsne.fit_transform(df_scaled)

# Plot
plt.figure(figsize=(7,5))
plt.scatter(X_embedded[:, 0], X_embedded[:, 1], c=y_kmeans, s=15, cmap="tab10")
plt.title("t-SNE Embedding")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.show()

> ✏️ **Question 10:** t-SNE shows well-separated groups. Does that mean these clusters are better than the PCA ones? Can we interpret the t-SNE axes like PCA loadings?

*Your answer:*

